# Learning how to Linear Regression on a housing dataset
Follow allong with the tutorial [Sklearn learning model for beginner](https://youtu.be/-IvNzmrcyUM?si=yFj89wcu9vC6wDk9) on Youtube.
The datas is coming from a real world [California Housing dataset](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset).

## Real world data with 8 features:
8 numeric, predictive attributes and the target
Attribute Information:
- `MedInc` median income in block group
- `HouseAge` median house age in block group
- `AveRooms` average number of rooms per household
- `AveBedrms` average number of bedrooms per household
- `Population` block group population
- `AveOccup` average number of household members
- `Latitude` block group latitude
- `Longitude` block group longitude



## Project setup
Add the project root to `sys.path` so we can import from `ml.projects.houses.*` modules.

In [1]:
import sys
from pathlib import Path

# Find project root (directory containing pyproject.toml)
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Project root: {root}")

Project root: /Users/Ugo.Arzur/dev/tests/ai/ml-playground


## Import dataset
Import the dataset from the default datasets of sklearn, using return_X_y flag to get a tuple.

In [2]:
from ml.projects.houses.data.loader import load_dataset

X, y = load_dataset()
print(X, y)

[[   8.3252       41.            6.98412698 ...    2.55555556
    37.88       -122.23      ]
 [   8.3014       21.            6.23813708 ...    2.10984183
    37.86       -122.22      ]
 [   7.2574       52.            8.28813559 ...    2.80225989
    37.85       -122.24      ]
 ...
 [   1.7          17.            5.20554273 ...    2.3256351
    39.43       -121.22      ]
 [   1.8672       18.            5.32951289 ...    2.12320917
    39.43       -121.32      ]
 [   2.3886       16.            5.25471698 ...    2.61698113
    39.37       -121.24      ]] [4.526 3.585 3.521 ... 0.923 0.847 0.894]


## Splitting the dataset
In order to train the model we need to split the dataset into a 80-20 percent for training and testing data.

This is acheived with the `train_test_split` function and the `test_size=0.2` parameter.

The `random_state=432` parameter allow us to shuffle the samples of data in a specific order instead of randomly (by default). Hence we can reproduce the exact workflow, it's like a seed in stable diffusion.

In [3]:
from ml.shared.data import split_dataset

X_train, X_test, y_train, y_test = split_dataset(X, y)
print(f"X 80% Training Data: {X_train}, {y_train}")
print(f"Y 20% Testing Data: {X_test}, {y_test}")

X 80% Training Data: [[   2.1442       52.            3.94886364 ...    2.61647727
    37.34       -121.88      ]
 [   4.0938       20.            5.66       ...    2.668
    38.35       -121.96      ]
 [   5.3925       51.            5.00677966 ...    2.54237288
    34.2        -118.23      ]
 ...
 [   4.3958       10.            6.15450644 ...    3.78540773
    34.09       -117.39      ]
 [   2.8125       29.            4.33367983 ...    2.1039501
    37.72       -122.15      ]
 [   3.4934       27.            5.46       ...    3.00666667
    37.57       -120.85      ]], [1.889 1.173 3.179 ... 1.307 1.574 1.938]
Y 20% Testing Data: [[   5.1039       52.            5.62803738 ...    2.43551402
    37.76       -122.23      ]
 [   2.5238       49.            4.19202363 ...    2.19645495
    37.8        -122.24      ]
 [   5.7393        9.            5.79227053 ...    3.03864734
    34.03       -117.31      ]
 ...
 [   3.5588       37.            4.26751592 ...    1.97770701
    37.97   

## Training model
We'll use [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression) to guess houses prices based on our dataset.

> Ordinary least squares Linear Regression.
> 
> LinearRegression fits a linear model with coefficients w = (w1, …, wp) to minimize the residual sum of squares between the observed targets in the dataset, and the targets predicted by the linear approximation.

### Description
The model will go sample by sample to compare each set of feature (X_train) to it's target (y_train).
This will make the model trained over our training data, but we want it to be able to predict the house price outside of our training data.

In [4]:
from ml.projects.houses.models.model_helper import train_model
from ml.shared.metrics import evaluate_model

model = train_model(X_train, y_train)  # LinearRegression by default

## Evaluating the model
We want to be sure the model learned data instead of memorizing it (overfitting).
For that we'll evaluate the model with the `x_test` feature part in order to have the `y_prediction`; once we have the `y_prediction`, we'll be able to compare top the values in `y_test`.

For the evaluation we use [r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html) which is a regression score function (best possible is 1.0)

In [5]:
r2 = evaluate_model(model, X_test, y_test)
print(r2)

# The accuracy score is ~0.60 so the model understands 60% of why the house price goes up and down.
# This is a baseline to improve it further

0.6080229586580346


The model is now 60% time correct when making a prediction.

## Polynomial Regression
A Polynomial feature allow to increase features in a regression.

With sklearn PolynomialFeatures, we can take our 8 -> 45 features and r2 score from 0.6 to 0.66 (0.2 increase).

## With 8 original features, it will create:
- All combinations (x₁·x₂, x₁·x₃, etc.)
- The squares (x₁², x₂², etc.)
- One constant (1)

## Useful
### 📊 Non linear relations
In real life a house price of 200m2 isn't the 100m2 price x2. The bigger the surface, different is the price.
### Features interactions
It will enrich the features when you think interactions are important: `MedInc (income) * HouseAge (age)`. This interaction is revealing that:
- in rich neighborhoods (MedInc high) and old houses (HouseAge high) is more valuable (charm)
- in poor neighborhoods (MedInc low) and old houses (HouseAge low) are less valuable (degraded)
This new feature `Medinc * HouseAge` can capture this!

### ⚠️ Warning
- The more you increase the `degree` parameter in `PolynomialFeatures()` the mode you have new columns.
- Can lead to overfitting if dataset is small.




In [6]:
from ml.projects.houses.features.polynomial import create_polynomial_features

print(f"Before Polynomial transformation {X.shape}")
X_poly, poly = create_polynomial_features(X)
print(f"After Polynomial transformation: {X_poly.shape}")

# Re-split with polynomial features
X_train, X_test, y_train, y_test = split_dataset(X_poly, y)

Before Polynomial transformation (20640, 8)
After Polynomial transformation: (20640, 45)


## Selecting the right model
The `HistGradientBoostingRegressor` and `RandomForestRegressor` models are good concurrent models and we'll try them with a regression score too.

In a loop of `[model, GBR, RFR]`:
1. we train the model on training data `X_train` and `y_train`
2. make prediction with `x_test`
3. score the regression with `y_test` and `y_pred`

This will return us multiple scores for multiple models.
```
Score for LinearRegression() is: 0.6080229586580346
Score for HistGradientBoostingRegressor() is: 0.8364438492135174
Score for RandomForestRegressor(n_jobs=-1) is: 0.8126642078111718
```

### Parameters
- `n_jobs=-1` is to use all cores available for the job, we can limit in positive numbers (1,3,10, etc)

In [7]:
from ml.projects.houses.models.model_helper import compare_models

results = compare_models(X_train, y_train, X_test, y_test)
for name, score in results.items():
    print(f"Score for {name} is: {score}")

Score for LinearRegression() is: 0.6610240211568321
Score for HistGradientBoostingRegressor() is: 0.836884008345191
Score for RandomForestRegressor(n_jobs=-1) is: 0.8042083046534848


## HyperParamertization : Parameter optimization
Now we'll evaluate what are the best parameters to use for our model.
How to do optimization ?
- First we experiment with the number of trees which is the `max_iter` parameter. In the `max_iteration_range` list we test every number.
- Then we search the learning rate

Results are
```
Starting Parameter optimization
Score for iteration of 250 with 0.1 is: 0.8434126795758516
Score for iteration of 300 with 0.1 is: 0.8426676580156846
Score for iteration of 350 with 0.1 is: 0.8439658387680059
Score for iteration of 250 with 0.05 is: 0.8375863801259467
Score for iteration of 300 with 0.05 is: 0.8411898616012664
Score for iteration of 350 with 0.05 is: 0.8448662786329125
Score for iteration of 250 with 0.001 is: 0.2608375545135623
Score for iteration of 300 with 0.001 is: 0.299112357206882
Score for iteration of 350 with 0.001 is: 0.33695141665669903
```
Which make the score for iteration of 350 with 0.05 the best at: 0.8448662786329125

In [8]:
from ml.projects.houses.models.model_helper import grid_search_hgbr

best_model, best_score, all_results = grid_search_hgbr(X_train, y_train, X_test, y_test)

print("Starting Parameter optimization")
for r in all_results:
    print(f"Score for iteration of {r['max_iter']} with {r['learning_rate']} is: {r['r2_score']}")
print(f"\nBest score: {best_score}")

Starting Parameter optimization
Score for iteration of 250 with 0.1 is: 0.8415630885082193
Score for iteration of 300 with 0.1 is: 0.841478955527572
Score for iteration of 350 with 0.1 is: 0.8436792683075421
Score for iteration of 250 with 0.05 is: 0.8388453898900288
Score for iteration of 300 with 0.05 is: 0.8430689657545866
Score for iteration of 350 with 0.05 is: 0.8429731492303123
Score for iteration of 250 with 0.001 is: 0.2632017965181803
Score for iteration of 300 with 0.001 is: 0.30327874117544684
Score for iteration of 350 with 0.001 is: 0.3412138843583318

Best score: 0.8436792683075421


## Exporting model
Now we have our perfect parameters, we want to export our model in order to save it on the file system, thanks to [joblib](https://joblib.readthedocs.io/en/stable/persistence.html#use-case).

In [9]:
from ml.projects.houses.models.model_helper import save_model

path = save_model(best_model, "hgbr_best.joblib")
print(f"Final model saved to {path}")

Final model saved to /Users/Ugo.Arzur/dev/tests/ai/ml-playground/ml/projects/houses/trained_models/hgbr_best.joblib


## Testing created model score
Now that our model has been trained and exported we can use it to predict on data.

The result should be:
`The score of the trained model is: 0.8436238636435226`

In [10]:
from ml.projects.houses.models.model_helper import load_model

trained_model = load_model("hgbr_best.joblib")
r2 = evaluate_model(trained_model, X_test, y_test)
print(f"The score of the trained model is: {r2}")

The score of the trained model is: 0.8436792683075421


# Predicting real price
Now let's predict real price of a house.

In [11]:
import numpy as np

trained_model = load_model("hgbr_best.joblib")

# [MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude]
house_example = np.array(
    [[8.3252, 41.0, 6.98, 1.02, 322.0, 2.55, 37.88, -122.23]]
)  # Ecological Study Area Oakland (SF)
house_example2 = np.array(
    [[4.4287, 25.0, 5.0, 3.0, 129.0, 2.55, 33.8347516, -117.911732]]
)  # Anaheim house

# Transform using the poly transformer from the polynomial features cell
house_example_poly = poly.transform(house_example)
house_example2_poly = poly.transform(house_example2)

predicted_price = trained_model.predict(house_example_poly)
predicted_price2 = trained_model.predict(house_example2_poly)
print(f"Predicted house 1 price: ${predicted_price[0] * 100000:.2f}")
print(f"Predicted house 2 price: ${predicted_price2[0] * 100000:.2f}")

Predicted house 1 price: $422863.23
Predicted house 2 price: $238490.81
